In [0]:
from pyspark.sql import functions as F
 
CATALOGO = "PUC_Sprint_2"
SCHEMA = "anp"
 
fact_producao = spark.table(f"{CATALOGO}.{SCHEMA}.fact_producao_mensal")
fact_reservas = spark.table(f"{CATALOGO}.{SCHEMA}.fact_reservas_anual")
fact_economico = spark.table(f"{CATALOGO}.{SCHEMA}.fact_economico")
dim_campo = spark.table(f"{CATALOGO}.{SCHEMA}.dim_campo")
dim_poco = spark.table(f"{CATALOGO}.{SCHEMA}.dim_poco")
dim_tempo = spark.table(f"{CATALOGO}.{SCHEMA}.dim_tempo")

Pergunta 1 (simples)
**Qual foi a produção total de óleo e gás natural por ano?**
**Lógica:** `fact_producao_mensal` já está no grão poço×mês, então basta somar as colunas de produção agrupando pelo `ano` de `dim_tempo` (join por `tempo_id`). Não precisamos de `dim_campo` nem `dim_poco` aqui - a pergunta é sobre o total do país, não por campo/poço.
 

In [0]:
resposta_1 = (fact_producao
    .join(dim_tempo, "tempo_id")
    .groupBy("ano")
    .agg(
        F.sum("producao_oleo_m3").alias("producao_oleo_m3"),
        F.sum(F.col("producao_gas_associado_mm3") + F.col("producao_gas_nao_associado_mm3")).alias("producao_gas_mm3"),
    )
    .orderBy("ano"))

resposta_1.show(100)

In [0]:
display(resposta_1)

Databricks visualization. Run in Databricks to view.

In [0]:
# 1. A Bronze de fato gravou 2024?
df_check = spark.table(f"{CATALOGO}.{SCHEMA}.bronze_bmp")
df_check.filter(F.col("Ano") == "2024").count()

In [0]:
# 1. Números reais dos últimos anos, sem gráfico no meio
resposta_1.orderBy(F.col("ano").desc()).show(10)

# 2. Teste: o join com dim_tempo está multiplicando linhas?
print("fact_producao_mensal original:", fact_producao.count())
print("fact_producao_mensal após join com dim_tempo:", fact_producao.join(dim_tempo, "tempo_id").count())

# 3. dim_tempo tem tempo_id duplicado? (mesma combinação ano+mês aparecendo 2x)
dim_tempo.groupBy("tempo_id").count().filter("count > 1").show()

In [0]:
spark.table(f"{CATALOGO}.{SCHEMA}.silver_bmp") \
    .select("mes_ano") \
    .distinct() \
    .orderBy(F.col("mes_ano").desc()) \
    .show(30, truncate=False)

In [0]:
# 1. Existe QUALQUER linha de 2024, em qualquer formato de texto?
spark.table(f"{CATALOGO}.{SCHEMA}.silver_bmp") \
    .filter(F.col("mes_ano").contains("2024")) \
    .select("mes_ano") \
    .distinct() \
    .show(20, truncate=False)

# 2. "dez/2025" e "12/2025" viram o mesmo tempo_id? (checando quantas linhas cada um tem)
spark.table(f"{CATALOGO}.{SCHEMA}.silver_bmp") \
    .filter(F.col("mes_ano").isin("dez/2025", "12/2025")) \
    .groupBy("mes_ano") \
    .count() \
    .show()

In [0]:
# Célula do Databricks - roda depois de subir os arquivos pro Volume
df_anual = spark.read.option("header", True).csv("/Volumes/PUC_Sprint_2/anp/raw/bmp-2024/producao_por_poco_2024.csv")
df_anual.printSchema()
df_anual.groupBy("mes_ano").count().show(20)

df_trim1 = spark.read.option("header", True).csv("/Volumes/PUC_Sprint_2/anp/raw/bmp-2024/producao-por-poco-terra-trim-1.csv")
df_trim1.printSchema()

In [0]:
df_anual.groupBy("Mês/Ano").count().orderBy("Mês/Ano").show(20)

In [0]:
df_anual.printSchema()

In [0]:
print(df_anual.columns)

In [0]:
colunas = df_anual.columns
coluna_mes = [c for c in colunas if "Ano" in c and "M" in c][0]  # pega a coluna certa sem precisar digitar o acento
print(f"Coluna encontrada: {coluna_mes!r}")

df_anual.groupBy(coluna_mes).count().orderBy(coluna_mes).show(20)

In [0]:
df_anual.groupBy("[Ambiente]").count().show()
df_anual.select("[Mês/Ano]").distinct().orderBy("[Mês/Ano]").count()  # deveria ser 12 se cobre o ano inteiro

In [0]:
for n in [1, 2, 3, 4]:
    caminho = f"/Volumes/PUC_Sprint_2/anp/raw/bmp-2024/producao-por-poco-terra-trim-{n}.csv" if n < 4 else \
              "/Volumes/PUC_Sprint_2/anp/raw/bmp-2024/producao_por_poco_terra_trim_4.csv"
    df_trim = spark.read.option("header", True).csv(caminho)
    meses = [r[0] for r in df_trim.select("[Mês/Ano]").distinct().orderBy("[Mês/Ano]").collect()]
    print(f"trim {n}: {df_trim.count()} linhas | meses: {meses}")